# Slice shifts: conditional distributions, searched and confirmed

Most statements in a report describe a column on its own, or a
relation between two columns over all rows. A slice statement
describes a column inside one subset of the rows: the share of
extreme values, the median, or a level's share inside the subset
differs from the table as a whole. "Within partner-channel
listings in Manhattan, 34% of prices sit above the table's 95th
percentile, against 5% overall" is a statement of this kind. The
group is not declared by the caller and is not a pair of columns:
it is a conjunction of conditions — categorical levels and
numeric thresholds — and the pass searches for it.

The `slice_shifts` detector is opt-in: request it with
`detectors=["slice_shifts"]`. A default profile records it as
skipped in coverage, naming it opt-in. Searching is the risk in
this statement kind: with thousands of candidate slices, some
slice shows a large shift by chance on any frame. The pass is
built around that risk, and the next section is the control stack
that answers it. Its findings are `associations` findings of type
`slice_shift`, with the form in the statement form: `tail_shift`
or `median_shift` for a numeric target, `level_share` for a
categorical one.

## How a slice is searched and confirmed

The pass runs on the shared sample (20,000 rows at the standard
budget, 50,000 at deep), split deterministically into two halves.
Conditions are levels of categorical columns with at most 25
distinct values on the sample (at most 12 condition columns,
keeping the most frequent levels) and thresholds on numeric
columns — the value above the column's 75th or 90th percentile.
A slice is one condition, or a conjunction of two: two levels of
different columns, or one level and one threshold. At the deep
budget, depth-2 slices that cleared the search bars are extended
by one further level, as a beam. A slice needs at least
max(50, 1% of the sample) rows, and at least 25 rows in each
half. A slice never conditions on its own target column.

On the search half, a slice × target pair becomes a candidate at
these bars: at least 15% of the slice's rows above the target's
global 95th percentile, or a slice median at least 0.75 IQR from
the global median; for a categorical level, a rate at least twice
or at most half the global rate, differing by at least 0.02, with
at least 5 expected positives. A candidate is stated only when
the same form reappears on the held-out half the search never
saw — tail share at least 10% and at least half the search-half
share, a median shift of at least 0.5 IQR
in the same direction, a rate ratio of at least 1.5 (at most 0.5
in the low direction; a search rate of exactly 0 confirms when
the held-out ratio is at most 0.25) — and its held-out p-value
clears Benjamini–Hochberg q ≤ 0.05 across the family of
candidates that cleared the search bars.

Three selection rules follow. Slices with identical row sets are
one statement, kept at the shallowest description. A deeper
slice whose rows are at least 90% the same as a shallower kept
slice for the same target is demoted as a restatement, not
removed. Statements are ranked by absolute effect with support
and capped at 20, at most 8 about one target column, so a large
ratio on a small absolute difference does not lead the report
and one target's near-restatements cannot fill it. The coverage entry states how many
slice × target candidates were searched, how many cleared the
search bars, and how many confirmed.

## Example: a planted tail inside a conjunction

Four thousand rows. The amount is one baseline distribution
everywhere, except inside segment B sold through the store
channel, where 40% of rows carry an extreme amount. No single
condition holds the shift: segment B alone dilutes it, the store
channel alone dilutes it further.

In [ ]:
import random

import polars as pl

import onceover

rng = random.Random(2721)
segment, channel, amount = [], [], []
for _ in range(4000):
    seg = rng.choice(["A", "B", "C", "D"])
    ch = rng.choice(["web", "store"])
    if seg == "B" and ch == "store" and rng.random() < 0.40:
        amount.append(rng.gauss(300.0, 20.0))
    else:
        amount.append(rng.gauss(50.0, 10.0))
    segment.append(seg)
    channel.append(ch)
df = pl.DataFrame({"segment": segment, "channel": channel, "amount": amount})

report = onceover.profile(df, detectors=["slice_shifts"])
shifts = report.select(type="slice_shift")
for f in shifts:
    print(f["claim"])
entry = report.coverage["slice_shifts"]
print(entry["status"], "-", entry["detail"])

## Read the finding

The claim states the full-sample figures: the slice's row count,
its tail share against the table-wide share, and its median
against the table-wide median. The evidence carries the three
readings of the statistic — search half, held-out half, full
sample — with the p-value of the held-out comparison and its
Benjamini–Hochberg q-value over the confirmation family, and the
candidate counts of the run: `candidates_evaluated` is every
slice × target pair the search scored; `search_candidates` is
the family that cleared the search bars and entered
confirmation.

In [ ]:
f = next(
    f for f in shifts
    if f["specification"]["form"] == "tail_shift"
    and len(f["specification"]["conditions"]) == 2
)
m = f["evidence"][0]["metrics"]
for key in (
    "conditions", "slice_rows", "tail_share", "tail_share_search",
    "tail_share_confirm", "global_tail_share", "slice_median",
    "global_median", "top1_share_in_slice", "row_share",
    "p_value", "q_value", "search_candidates", "candidates_evaluated",
):
    print(key, "=", m[key])
print("validation:", f["evidence"][0]["validation"])

`top1_share_in_slice` against `row_share` is the share of the
column's top-1% values the slice carries, relative to its share
of the rows. When the slice holds at least 3 times its row share
of the top-1% values, and at least 8 such values, the claim says
so in a second clause: a small slice carrying most of a column's
extremes is part of what the statement means, not a separate
finding type.

## Example: a level share

The same search runs against categorical targets, where the
statistic is a level's share inside the slice against its
table-wide rate.

In [ ]:
rng = random.Random(96)
region, subscribed = [], []
for _ in range(4000):
    reg = rng.choice(["north", "south", "east", "west"])
    p = 0.45 if reg == "south" else 0.08
    subscribed.append("yes" if rng.random() < p else "no")
    region.append(reg)
df2 = pl.DataFrame({"region": region, "subscribed": subscribed})

report2 = onceover.profile(df2, detectors=["slice_shifts"])
for f in report2.select(type="slice_shift"):
    print(f["claim"])

## The negative control

Destroy the association — shuffle the target column of the first
frame, leaving the slice columns untouched — and the same search
states nothing. The split-half confirmation is what the shuffled
frame cannot pass: a shift found by searching one half does not
reappear, at the confirmation bars, on the other.

In [ ]:
amounts = df["amount"].to_list()
random.Random(7).shuffle(amounts)
shuffled = df.with_columns(pl.Series("amount", amounts))
report3 = onceover.profile(shuffled, detectors=["slice_shifts"])
print("statements:", len(report3.select(type="slice_shift")))
print(report3.coverage["slice_shifts"]["detail"])

## Data and work limits

- The search runs on the shared sample: 20,000 rows at the
  standard budget, 50,000 at deep; the profile's `sample_rows`
  override moves it with the other passes. Statistics, quantiles,
  and the split all refer to the sample, and the coverage entry
  states its size and both halves.
- At most 12 categorical condition columns, 12 threshold columns,
  24 numeric targets, and 8 categorical target columns take part;
  categorical columns need between 2 and 25 distinct values on
  the sample, and numeric threshold columns at least 20 distinct
  values. Caps that bind are stated in the coverage entry.
- The search stops at a work budget of 300 million cell visits
  (or at the profile deadline, polled between slices). A stopped
  search reports `capped` coverage and states how many of the
  slice definitions were examined: the examined set is a prefix
  of the deterministic enumeration, so a stopped run's silence is
  a partial negative, not an exhaustive one.
- At most 20 statements are made, and at most 8 about one
  target column; confirmed slices past the caps are counted in
  coverage, not stated.

## Interpretation

- A slice statement is a searched association, not a cause, and
  the evidence is the confirmation record, not a recount. Read
  the held-out figures before the full-sample ones: a statement
  whose confirmation half barely clears its bar is weaker than
  its claim's full-sample numbers look.
- A slice can be definitional: the distribution of age inside a
  retired level differs because of what the level means. Such
  statements are true and replicable, and the pass does not
  suppress them; they compete on effect and support with every
  other statement, and the interpretation is the reader's work.
- Rows missing a condition value are outside the slice; rows
  missing the target count as not extreme and not positive, so
  shares are computed against all of the slice's rows.
- `report.verify` marks slice statements `not_checkable`: the
  statistics are measured on the pass's sample and split, which a
  recount on the full frame reproduces neither. The slice
  definition is carried as structured predicates in the statement
  definition, so the slice itself can be recounted on any row set
  directly.

## Checks

The implementation is tested on planted fixtures of each form —
a depth-2 tail slice, a depth-1 median shift, a level share, a
rate of exactly 0 confirmed under the zero-tolerant rule — on a
shuffled-target frame, which must state nothing, and on two
identical condition columns, whose identical row sets must
produce one statement. The acceptance measurements against the
prototype protocol this design was measured with, on six
datasets, are recorded in
`docs/benchmarks/2026-11-13-slice-shifts.md`.